# 01 · Limpieza del catálogo

**Objetivo.** Construir el catálogo de trabajo a partir de el archivo de Data_raw 

**Entrada:** `data_raw/nasa_exoplanets_raw_2026_07_21.csv` (copia original).

**Salidas:**
- `data_processed/01_variables_seleccionadas.csv`
- `data_processed/02_catalogo_limpio_base.csv`
- Tabla D-T1 (depuración) y tabla D-T2 (variables y disponibilidad)

**Criterios de inclusión:** planetas con nombre y estrella anfitriona identificables.

**Criterios de exclusión:** planetas duplicados por nombre y planetas marcados como controversiales (`pl_controv_flag = 1`). No se sustituyen valores faltantes.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from funciones import *

usar_estilo()

## 1. Catálogo original

In [7]:
df_raw = cargar_catalogo_crudo()
print("Filas y columnas del catálogo original:", df_raw.shape)
print("Planetas:", df_raw["pl_name"].nunique(), "| estrellas anfitrionas:", df_raw["hostname"].nunique())

Filas y columnas del catálogo original: (6324, 127)
Planetas: 6324 | estrellas anfitrionas: 4738


## 2. Selección de variables y depuración

La lista de variables está en `funciones.py` (`COLUMNAS_TRABAJO`). Además de las variables físicas se conservan los indicadores de límite de radio y masa (`pl_radelim`, `pl_bmasselim`), que se necesitan en el análisis II.

In [8]:
df_work, df_clean, tabla_depuracion = limpiar_catalogo(df_raw)

print("Filas duplicadas completas:", df_work.duplicated().sum())
print("Planetas sin nombre:", df_work["pl_name"].isna().sum(), "| sin estrella:", df_work["hostname"].isna().sum())

revision = revisar_valores_fisicos(df_clean)
print("Valores físicamente imposibles (<= 0):", revision.iloc[:, 1].sum())
revision

Filas duplicadas completas: 0
Planetas sin nombre: 0 | sin estrella: 0
Valores físicamente imposibles (<= 0): 0


,variable,N_valores_menores_o_iguales_a_cero
0,pl_rade,0
1,pl_bmasse,0
2,pl_dens,0
3,pl_orbper,0
4,pl_orbsmax,0
5,st_teff,0
6,st_mass,0
7,st_rad,0
8,sy_dist,0


In [9]:
tabla_depuracion = guardar_tabla(
    tabla_depuracion, "D-T1", "depuracion_catalogo",
    titulo="Depuración del catálogo: número de planetas y de estrellas anfitrionas después de cada criterio.",
    seccion="6.1.1", destino="anexo",
    nota="No se encontraron duplicados ni valores menores o iguales a cero en las variables físicas.")
tabla_depuracion

Tabla D-T1 guardada: tablas/D-T1_depuracion_catalogo.csv


,paso,N_planetas,N_estrellas,N_excluidos
0,Catálogo original (PSCompPars),6324,4738,0
1,Sin planetas duplicados por nombre,6324,4738,0
2,Sin planetas controversiales (pl_controv_flag ...,6267,4711,57


## 3. Variables seleccionadas y disponibilidad

La disponibilidad se calcula sobre el catálogo limpio, que es el que usan todos los análisis.

In [10]:
variables = tabla_variables(df_clean)
variables = guardar_tabla(
    variables, "D-T2", "variables_y_disponibilidad",
    titulo="Variables seleccionadas del catálogo, unidades y disponibilidad en el catálogo limpio.",
    
    seccion="6.1.1", destino="anexo", decimales=2,
    encabezados={"datos_disponibles": "Disponibles", "datos_faltantes": "Faltantes",
                 "porcentaje_faltante": "Faltante (\\%)", "tipo": "Tipo"})
variables.sort_values("porcentaje_faltante", ascending=False).head(12)

Tabla D-T2 guardada: tablas/D-T2_variables_y_disponibilidad.csv


,variable,descripcion,unidad,tipo,datos_disponibles,datos_faltantes,porcentaje_faltante
27,st_spectype,Tipo espectral,,texto,2303,3964,63.25
35,st_age,Edad estelar,Gyr,decimal,4853,1414,22.56
18,pl_orbeccen,Excentricidad,adimensional,decimal,5225,1042,16.63
32,st_metratio,Razón usada para la metalicidad,,texto,5622,645,10.29
31,st_met,Metalicidad estelar,dex,decimal,5639,628,10.02
25,pl_insol,Flujo incidente reportado,flujo terrestre,decimal,5695,572,9.13
26,pl_eqt,Temperatura de equilibrio,K,decimal,5754,513,8.19
17,pl_orbsmax,Semieje mayor,au,decimal,5846,421,6.72
16,pl_orbper,Período orbital,días,decimal,5927,340,5.42
34,st_logg,Gravedad superficial estelar,log10(cm/s^2),decimal,5947,320,5.11


## 4. Guardar el catálogo de trabajo

In [11]:
df_work.to_csv(CATALOGO_SELECCION, index=False)
df_clean.to_csv(CATALOGO_LIMPIO, index=False)
print("Catálogo con variables seleccionadas:", df_work.shape)
print("Catálogo limpio:", df_clean.shape)

Catálogo con variables seleccionadas: (6324, 37)
Catálogo limpio: (6267, 37)
